In [1]:
import json
import yaml
from pathlib import Path
from statistics import mean
import pandas as pd

def load_yaml(path: str | Path) -> dict:
    """Memuat file YAML dan mengembalikan dictionary kosong jika file tidak valid."""
    filepath = Path(path)
    if not filepath.is_file():
        raise FileNotFoundError(f"File konfigurasi tidak ditemukan: {filepath}")
    
    with open(filepath, "r", encoding="utf-8") as f:
        return yaml.safe_load(f) or {}

def load_json(path: str | Path) -> dict | list:
    """Memuat file JSON."""
    filepath = Path(path)
    if not filepath.is_file():
        raise FileNotFoundError(f"File JSON tidak ditemukan: {filepath}")
        
    with open(filepath, "r", encoding="utf-8") as f:
        return json.load(f)

def load_jsonl(path: str | Path) -> dict | list:
    """Memuat file JSON."""
    filepath = Path(path)
    if not filepath.is_file():
        raise FileNotFoundError(f"File JSON tidak ditemukan: {filepath}")
        
    with open(path, "r", encoding="utf-8") as f:
        data = [json.loads(line) for line in f if line.strip()]

    return data

        
def get_rouge_score(method_num: str, path: str) -> dict:
    """Memuat skor ROUGE dari path yang ditentukan."""
    new_path = Path(f"{method_num}_outputs") / path / "rouge.json"
    return load_json(new_path)

def reformat_unieval(unieval: list) -> dict:
    """Menghitung nilai rata-rata untuk setiap metrik UniEval."""
    if not unieval:
        return {}
        
    coherence = [item.get('coherence', 0) for item in unieval]
    relevance = [item.get('relevance', 0) for item in unieval]
    fluency = [item.get('fluency', 0) for item in unieval]
    consistency = [item.get('consistency', 0) for item in unieval]

    return {
        "coherence": mean(coherence),
        "relevance": mean(relevance),
        "fluency": mean(fluency),
        "consistency": mean(consistency)
    }

def reformat_abstractiveness(abstr: list) -> dict:
    """Menghitung nilai rata-rata untuk setiap metrik abstraksi."""
    if not abstr:
        return {}

    novel_1gram = [it['novel_1gram'] for it in abstr if it.get('novel_1gram') is not None]
    novel_2gram = [it['novel_2gram'] for it in abstr if it.get('novel_2gram') is not None]
    novel_3gram = [it['novel_3gram'] for it in abstr if it.get('novel_3gram') is not None]
    novel_4gram = [it['novel_4gram'] for it in abstr if it.get('novel_4gram') is not None]
    coverage    = [it['coverage']    for it in abstr if it.get('coverage')    is not None]
    density     = [it['density']     for it in abstr if it.get('density')     is not None]

    return {
        "novel_1gram": mean(novel_1gram) if novel_1gram else 0,
        "novel_2gram": mean(novel_2gram) if novel_2gram else 0,
        "novel_3gram": mean(novel_3gram) if novel_3gram else 0,
        "novel_4gram": mean(novel_4gram) if novel_4gram else 0,
        "coverage":    mean(coverage)    if coverage    else 0,
        "density":     mean(density)     if density     else 0,
    }


def reformat_format_metrics(fmt: list) -> dict:
    """Menghitung nilai rata-rata untuk setiap metrik kepatuhan format."""
    if not fmt:
        return {}

    has_user_profile_strict = [it['has_user_profile_strict'] for it in fmt if it.get('has_user_profile_strict') is not None]
    has_profile_any         = [it['has_profile_any']         for it in fmt if it.get('has_profile_any')         is not None]
    has_problem             = [it['has_problem']             for it in fmt if it.get('has_problem')             is not None]
    has_resolution          = [it['has_resolution']          for it in fmt if it.get('has_resolution')          is not None]
    has_all_three           = [it['has_all_three']           for it in fmt if it.get('has_all_three')           is not None]
    n_bullets               = [it['n_bullets']               for it in fmt if it.get('n_bullets')               is not None]
    bullets_violation       = [it['bullets_violation']       for it in fmt if it.get('bullets_violation')       is not None]
    is_truncated            = [it['is_truncated']            for it in fmt if it.get('is_truncated')            is not None]
    has_boilerplate         = [it['has_boilerplate']         for it in fmt if it.get('has_boilerplate')         is not None]
    n_words                 = [it['n_words']                 for it in fmt if it.get('n_words')                 is not None]
    n_chars                 = [it['n_chars']                 for it in fmt if it.get('n_chars')                 is not None]

    return {
        "has_user_profile_strict": mean(has_user_profile_strict) if has_user_profile_strict else 0,
        "has_profile_any":         mean(has_profile_any)         if has_profile_any         else 0,
        "has_problem":             mean(has_problem)             if has_problem             else 0,
        "has_resolution":          mean(has_resolution)          if has_resolution          else 0,
        "has_all_three":           mean(has_all_three)           if has_all_three           else 0,
        "n_bullets":               mean(n_bullets)               if n_bullets               else 0,
        "bullets_violation":       mean(bullets_violation)       if bullets_violation       else 0,
        "is_truncated":            mean(is_truncated)            if is_truncated            else 0,
        "has_boilerplate":         mean(has_boilerplate)         if has_boilerplate         else 0,
        "n_words":                 mean(n_words)                 if n_words                 else 0,
        "n_chars":                 mean(n_chars)                 if n_chars                 else 0,
    }


def reformat_salience_score(salience_score: list) -> dict:
    """Menghitung nilai rata-rata untuk setiap metrik UniEval."""
    if not salience_score:
        return {}
        
    slot_precision = [
        item.get("slot_precision") if item.get("slot_precision") is not None else 0
        for item in salience_score
    ]
    slot_recall = [
        item.get("slot_recall") if item.get("slot_recall") is not None else 0
        for item in salience_score
    ]
    slot_f1 = [
        item.get("slot_f1") if item.get("slot_f1") is not None else 0
        for item in salience_score
    ]
    scr = [item.get("scr") if item.get("scr") is not None else 0 for item in salience_score]

    return {
        "slot_precision": mean(slot_precision),
        "slot_recall": mean(slot_recall),
        "slot_f1": mean(slot_f1),
        "scr": mean(scr)
    }
    
def get_unieval_score(method_num: str, path: str) -> dict:
    """Memuat skor UniEval dari path yang ditentukan lalu merapihkannya."""
    new_path = Path(f"{method_num}_outputs") / path / "unieval.json"
    unieval_data = load_json(new_path)
    return reformat_unieval(unieval_data)
    # Menampilkan DataFrame

def get_align_score(method_num:str, path: str) -> dict:
    new_path = Path(f"{method_num}_outputs") / path / "alignscore.json"
    align_score_data = load_json(new_path)
    return { "align_score" : mean(align_score_data)}

def get_salience_score(method_num: str, path: str) -> dict:
    new_path = Path(f"{method_num}_outputs") / path / "saliencescore.jsonl"
    align_score_data = load_jsonl(new_path)
    
    return reformat_salience_score(align_score_data)

def get_cam_score(method_num: str, path: str) -> dict:
    new_path = Path(f"{method_num}_outputs") / path / "cam.jsonl"
    align_score_data = load_jsonl(new_path)
    
    return reformat_abstractiveness(align_score_data)

def get_cfm_score(method_num: str, path: str) -> dict:
    new_path = Path(f"{method_num}_outputs") / path / "cfm.jsonl"
    align_score_data = load_jsonl(new_path)
    
    return reformat_format_metrics(align_score_data)

# Setup konfigurasi
def read_result(method_num):
    method_cfg = load_yaml(f"configs/methods/{method_num}.yaml")
    
    dataset_path = method_cfg.get('datasets')
    if dataset_path:
        dataset_cfg = load_yaml(f"configs/datasets/{dataset_path}")
    
    # Memuat list eksperimen
    experiments = method_cfg.get('experiments', [])
    
    # Mengumpulkan nama folder output dari tiap eksperimen
    
    # Jika ingin langsung mengambil skor metrik, Anda bisa melakukan iterasi pada folder_list:
    for path in folder_list:
    
        rouge = get_rouge_score(method_num, path)
        unieval_mean = get_unieval_score(method_num, path)
        align_score_mean = get_align_score(method_num, path)
        cfm = get_cfm_score(method_num, path)
        cam = get_cam_score(method_num, path)
        rows = {
                **unieval_mean,
                **align_score_mean,
                **cam,
                **cfm            
        }
    df = pd.DataFrame(rows)
    # Mengubah list of dictionaries menjadi DataFrame
    return df

In [2]:
# Setup konfigurasi
method_num = "method2"
def main(method_num):
    method_cfg = load_yaml(f"configs/methods/{method_num}.yaml")
    
    dataset_path = method_cfg.get('datasets')
    if dataset_path:
        dataset_cfg = load_yaml(f"configs/datasets/{dataset_path}")
    
    # Memuat list eksperimen
    experiments = method_cfg.get('experiments', [])
    
    # Mengumpulkan nama folder output dari tiap eksperimen
    folder_list = []
    for exp in experiments:
        exp_code = exp.get('code')
        if exp_code:
            exp_cfg = load_yaml(f"configs/experiments/{exp_code}")
            folder_list.append(exp_cfg.get('output_name'))
    
    # Jika ingin langsung mengambil skor metrik, Anda bisa melakukan iterasi pada folder_list:
    all_results = []
    
    for folder in folder_list:
        rouge = get_rouge_score(method_num, folder)
        unieval_mean = get_unieval_score(method_num, folder)
        align_score_mean = get_align_score(method_num, folder)
        salience = get_salience_score(method_num, folder)
        cfm = get_cfm_score(method_num, folder)
        cam = get_cam_score(method_num, folder)
        
        # Menggabungkan dictionary menggunakan operator unpack (**)
        # Kolom 'nama_file' ditambahkan secara eksplisit
        row_data = {
            'nama_file': folder,
            **rouge,
            **unieval_mean,
            **align_score_mean,
            **salience,
            **cfm,
            **cam
        }
        
        all_results.append(row_data)

    # Mengubah list of dictionaries menjadi DataFrame
    df = pd.DataFrame(all_results)
    return df

In [3]:
#600 data / 5 epoch 
df1 = main('method1')
df1['nama_file'] = df1['nama_file'] + '_E1'
df1

,nama_file,rouge1,rouge2,rougeL,rougeLsum,bertscore_precision,bertscore_recall,bertscore_f1,coherence,relevance,...,is_truncated,has_boilerplate,n_words,n_chars,novel_1gram,novel_2gram,novel_3gram,novel_4gram,coverage,density
0,bart_finetune_E1,0.594181,0.316315,0.384190,0.559028,0.915067,0.902619,0.908725,0.820494,0.767995,...,0.220,0,83.084,534.916,0.371327,0.706761,0.840957,0.912296,0.726497,2.021255
1,bart_margin_part_E1,0.566213,0.288011,0.365452,0.534488,0.912901,0.897118,0.904859,0.825707,0.755708,...,0.208,0,77.342,497.174,0.364515,0.688511,0.826514,0.901837,0.722387,2.118095
2,bart_margin_full_E1,0.574087,0.294008,0.364836,0.541145,0.912482,0.898647,0.905435,0.812972,0.749460,...,0.156,0,81.200,519.396,0.362988,0.687292,0.824878,0.900953,0.725340,2.170143


In [4]:
# 2000 data 5 epoch
df2 = main('method2')
df2['nama_file'] = df2['nama_file'] + '_E2'
df2

,nama_file,rouge1,rouge2,rougeL,rougeLsum,bertscore_precision,bertscore_recall,bertscore_f1,coherence,relevance,...,is_truncated,has_boilerplate,n_words,n_chars,novel_1gram,novel_2gram,novel_3gram,novel_4gram,coverage,density
0,bart_warmup_E2,0.605962,0.329708,0.401549,0.572090,0.915775,0.906013,0.910804,0.839601,0.794868,...,0.338,0,85.320,550.584,0.383111,0.721034,0.853717,0.923121,0.718113,1.919353
1,bart_finetune_E2,0.613364,0.332924,0.410316,0.579084,0.915275,0.906715,0.910903,0.851236,0.821201,...,0.382,0,86.680,560.664,0.393050,0.735960,0.864072,0.929696,0.710433,1.816670
2,bart_margin_part_E2,0.605501,0.321662,0.396590,0.570969,0.914256,0.903388,0.908723,0.849647,0.821132,...,0.300,0,82.858,537.184,0.389258,0.725743,0.857779,0.925483,0.707900,1.868712
3,bart_margin_full_E2,0.607991,0.325413,0.399937,0.572669,0.914600,0.904954,0.909686,0.867692,0.841028,...,0.278,0,84.372,548.240,0.389687,0.724492,0.856604,0.924974,0.708833,1.877980


In [5]:
# 2000 data / 10 epoch

df3 = main('method3')
df3['nama_file'] = df3['nama_file'] + '_E3'
df3

,nama_file,rouge1,rouge2,rougeL,rougeLsum,bertscore_precision,bertscore_recall,bertscore_f1,coherence,relevance,...,is_truncated,has_boilerplate,n_words,n_chars,novel_1gram,novel_2gram,novel_3gram,novel_4gram,coverage,density
0,bart_warmup_E3,0.613390,0.333760,0.410889,0.577780,0.915424,0.907573,0.911411,0.862478,0.824904,...,0.382,0,87.888,567.532,0.396547,0.737102,0.865096,0.930951,0.707596,1.796734
1,bart_finetune_E3,0.621431,0.342024,0.423796,0.586851,0.914764,0.910108,0.912370,0.879601,0.841998,...,0.412,0,91.242,589.936,0.408898,0.748918,0.872897,0.935126,0.696825,1.735517
2,bart_margin_part_E3,0.615583,0.332887,0.415667,0.580536,0.915093,0.906642,0.910773,0.888483,0.858750,...,0.278,0,86.136,561.504,0.398664,0.738140,0.867490,0.931848,0.699601,1.791228
3,bart_margin_full_E3,0.619879,0.334087,0.419140,0.583007,0.914060,0.908244,0.911078,0.895163,0.860808,...,0.386,0,88.814,580.080,0.408101,0.743338,0.869725,0.933830,0.693108,1.755160


In [7]:
df6 = main('method6')
df6['nama_file'] = df6['nama_file'] + '_E6'
df6

,nama_file,rouge1,rouge2,rougeL,rougeLsum,bertscore_precision,bertscore_recall,bertscore_f1,coherence,relevance,...,is_truncated,has_boilerplate,n_words,n_chars,novel_1gram,novel_2gram,novel_3gram,novel_4gram,coverage,density
0,qwen_5_warmup_E6,0.478118,0.227992,0.339757,0.444484,0.875951,0.890697,0.883030,0.870425,0.846404,...,0.188,0,117.928,837.106,0.634005,0.866807,0.940144,0.972336,0.478351,1.047349
1,qwen_5_finetune_E6,0.574615,0.300675,0.421318,0.542876,0.884937,0.913387,0.898577,0.848703,0.816714,...,0.212,0,123.496,915.484,0.567086,0.834092,0.924071,0.963870,0.569720,1.325614
2,qwen_5_margin_part_E6,0.547707,0.284932,0.405210,0.518149,0.896843,0.898280,0.897276,0.884386,0.844489,...,0.812,0,87.688,614.170,0.544316,0.829311,0.923531,0.964348,0.571428,1.299096
3,qwen_5_margin_full_E6,0.563900,0.289938,0.409560,0.532998,0.899102,0.902301,0.900541,0.887009,0.842304,...,0.800,0,93.020,644.550,0.544276,0.829022,0.921513,0.962661,0.576494,1.322021
4,qwen_5_base_E6,0.219712,0.047312,0.123550,0.199574,0.818383,0.838972,0.828214,0.756311,0.687623,...,0.318,0,172.872,1243.694,0.799420,0.950082,0.980414,0.990968,0.261490,0.473502


In [ ]:
df4 = main('method4')
df4['nama_file'] = df4['nama_file'] + '_E4'
df4

In [ ]:
df_bart = pd.concat([df1, df2, df3, df4])
df_bart.to_excel("bart_summary.xlsx")

df_arch = pd.concat([df4, df6])
df_arch.to_excel("arch_summary.xlsx")

In [ ]:
df9 = main('method9')
df9['nama_file'] = df9['nama_file'] + '_9'
df9

In [ ]:
rouge_test = get_rouge_score('method9', 'bart_margin_part/test')
rouge_test

In [ ]:
unieval = get_unieval_score('method9', 'bart_margin_part/test')
unieval

In [ ]:
align_score = get_align_score('method9', 'bart_margin_part/test')
align_score

In [ ]:
df_concat = pd.concat([df4, df6])
df_concat

In [ ]:
df_concat['mean'] = df_concat[['rouge1','rouge2','rougeL','rougeLsum','bertscore_precision','bertscore_recall','bertscore_f1','coherence','relevance','fluency','consistency','align_score']].mean(axis=1)

In [ ]:
df_concat

In [ ]:
def sort_values_by_metric(df, metric):
    return df.sort_values(by=metric, ascending=False)['nama_file']

In [ ]:
print(sort_values_by_metric(df_concat, 'align_score'))

In [ ]:
print(sort_values_by_metric(df_concat, 'consistency'))

In [ ]:
print(sort_values_by_metric(df_concat, 'relevance'))

In [ ]:
print(sort_values_by_metric(df_concat, 'coherence'))